# Comparison of the annotators

## Annotator vs GOAT
COmpare how much the annotated images different from the GAOT prediction used as a baseline.

In [ ]:
# imports
import os
import sys
from typing import List, Tuple
from tqdm import tqdm
from utils import compute_iou_matrix, instance_matcher


sys.path.append(os.path.dirname(os.getcwd()))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.model.model import maskRCNNModel
from src.model.dataset import MaskRCNNDataset
from src.utils.data_utils import *
from src.utils.annotation_utils import *
from src.utils.utils import *
from src.utils.const import *



In [ ]:
# dataset_1 = "../dataset/merge_alone_1_corrected_LT"
# dataset_2 = "../dataset/merge_alone_1_corrected_EK"

# dataset_1 = "../dataset/merge_alone_1_LT"
# dataset_2 = "../dataset/merge_alone_1_EK"

dataset_1 = "../dataset/Ekin_m"
dataset_2 = "../dataset/Lucrezia_m"

dataset_1 = MaskRCNNDataset(dataset_1, datatype="eval", data_augmentation=False)
dataset_2 = MaskRCNNDataset(dataset_2, datatype="eval", data_augmentation=False)

print(f"Dataset 1 length: {len(dataset_1)}, average objects per image: {np.mean([len(dataset_1[i][1]['boxes']) for i in range(len(dataset_1))])}")
print(f"Dataset 2 length: {len(dataset_2)}, average objects per image: {np.mean([len(dataset_2[i][1]['boxes']) for i in range(len(dataset_2))])}")

## Comapre the two annotators one wrt to the other.

Next we want to avaluate the inter-rater reliability. To do so we compute the precision and recall for varying iou thresholds when considering one of the annotators the ground truth and the opposite.

More specifically, inca se ann1 is the ground turht and ann2 the prediciton, we call:

- **True Positive**: the predictions of ann2 that have been matched to one prediction of ann1 (remark: these associations are unique and reversible).
- **False Positives**: the instances of ann2 that have no matching among the instances from ann1 (the ground truth)
- **False Negatives**: the instances of ann1 (the ground truth) taht have no matching among the instances of ann2

**REMARK**: note that when we invert the ground truth role the TP remanin unchanged (due to the revesibility) and the FP and FN swap.

In [ ]:
from copy import deepcopy

iou_threshold = np.linspace(0.1, 0.9, 20)

precisions_1 = []
recalls_1 = []
precisions_2 = []
recalls_2 = []

for i in range(len(dataset_1)):
    bboxes_annotated_1 = dataset_1[i][1]['boxes'].numpy()
    bboxes_annotated_2 = dataset_2[i][1]['boxes'].numpy()
    if len(bboxes_annotated_1) == 0 or len(bboxes_annotated_2) == 0:
        print(f"One of the annotations is empty. Skipping the image {i}. \nNr. annotated 1: {len(bboxes_annotated_1)} \nNr. annotated 2: {len(bboxes_annotated_2)}")
        continue

    # compute the iou matrix
    iou_matrix = compute_iou_matrix(bboxes_annotated_1, bboxes_annotated_2)
    precisions_1_image = []
    recalls_1_image = []
    precisions_2_image = []
    recalls_2_image = []
    for iou_th in iou_threshold:
        row_ind, col_ind = instance_matcher(deepcopy(iou_matrix), iou_th)
        row_ind_2, col_ind_2 = instance_matcher(deepcopy(iou_matrix.T), iou_th)
        pairs = list(zip(row_ind, col_ind))
        pairs_2 = list(zip(col_ind_2, row_ind_2))
        assert len(pairs) == len(pairs_2), "The number of pairs should be equal."
        for pair in pairs:
            assert pair in pairs_2, f"The pairs should be equal. \nPair: \n{pair} \nPairs_2: \n{pairs_2}"
        # compute the true positives
        true_positives = len(row_ind)
        # compute the false positives
        false_positives_1 = len(bboxes_annotated_1) - len(row_ind)
        false_positives_2= len(bboxes_annotated_2) - len(row_ind)
        # compute the false negatives
        false_negatives_1 = len(bboxes_annotated_2) - len(row_ind)
        false_negatives_2 = len(bboxes_annotated_1) - len(row_ind)
        # compute the precision
        precisions_1_image.append(true_positives / (true_positives + false_positives_1))
        precisions_2_image.append(true_positives / (true_positives + false_positives_2))
        # compute the recall
        recalls_1_image.append(true_positives / (true_positives + false_negatives_1))
        recalls_2_image.append(true_positives / (true_positives + false_negatives_2))
    precisions_1.append(precisions_1_image)
    recalls_1.append(recalls_1_image)
    precisions_2.append(precisions_2_image)
    recalls_2.append(recalls_2_image)
precisions_1 = np.array(precisions_1)
print(precisions_1.shape)
recalls_1 = np.array(recalls_1)
precisions_2 = np.array(precisions_2)
recalls_2 = np.array(recalls_2)



In [ ]:
recalls_1_mean = np.mean(recalls_1, axis=0)
recalls_1_std = np.std(recalls_1, axis=0)
precisions_1_mean = np.mean(precisions_1, axis=0)
precisions_1_std = np.std(precisions_1, axis=0)
recalls_2_mean = np.mean(recalls_2, axis=0)
recalls_2_std = np.std(recalls_2, axis=0)
precisions_2_mean = np.mean(precisions_2, axis=0)
precisions_2_std = np.std(precisions_2, axis=0)

print(f"Mean precision Annotator 1: {precisions_1_mean}")
print(f"Mean recall Annotator 1: {recalls_1_mean}")
print(f"Mean precision Annotator 2: {precisions_2_mean}")
print(f"Mean recall Annotator 2: {recalls_2_mean}")

# recall plot
fig = plt.subplots(figsize =(12, 8)) 
plt.plot(iou_threshold, recalls_1_mean, label="Ann. 1", marker="o", linestyle="--")
plt.fill_between(iou_threshold, recalls_1_mean - recalls_1_std, recalls_1_mean + recalls_1_std, alpha=0.2)
plt.plot(iou_threshold, recalls_2_mean, label="Ann. 2", marker="o", linestyle="--")
plt.fill_between(iou_threshold, recalls_2_mean - recalls_2_std, recalls_2_mean + recalls_2_std, alpha=0.2)
plt.xlabel("IoU Threshold", fontweight ='bold', fontsize = 12)
plt.ylabel("Recall", fontweight ='bold', fontsize = 12)
plt.title("Recall comparison", fontweight ='bold', fontsize = 15)
plt.legend()
plt.show()

# precision plot
fig = plt.subplots(figsize =(12, 8)) 
plt.plot(iou_threshold, precisions_1_mean, label="Ann. 1", marker="o", linestyle="--")
plt.fill_between(iou_threshold, precisions_1_mean - precisions_1_std, precisions_1_mean + precisions_1_std, alpha=0.2)
plt.plot(iou_threshold, precisions_2_mean, label="Ann. 2", marker="o", linestyle="--")
plt.fill_between(iou_threshold, precisions_2_mean - precisions_2_std, precisions_2_mean + precisions_2_std, alpha=0.2)
plt.xlabel("IoU Threshold", fontweight ='bold', fontsize = 12)
plt.ylabel("Precision", fontweight ='bold', fontsize = 12)
plt.title("Precision comparison", fontweight ='bold', fontsize = 15)
plt.legend()
plt.ylim(0, 1)
plt.show()

# precision_recall_plot

fig = plt.subplots(figsize =(12, 8))
plt.plot(recalls_1_mean, precisions_1_mean, label="Ann. 1", marker="o", linestyle="--")
plt.plot(recalls_2_mean, precisions_2_mean, label="Ann. 2", marker="o", linestyle="--")
plt.xlabel("Recall", fontweight ='bold', fontsize = 12)
plt.ylabel("Precision", fontweight ='bold', fontsize = 12)
plt.title("Precision-Recall comparison", fontweight ='bold', fontsize = 15)
plt.legend()
plt.ylim(0, 1)
plt.show()

# F1 score
f1_1 = 2 * precisions_1_mean * recalls_1_mean / (precisions_1_mean + recalls_1_mean)
f1_2 = 2 * precisions_2_mean * recalls_2_mean / (precisions_2_mean + recalls_2_mean)
fig = plt.subplots(figsize =(12, 8))
plt.plot(iou_threshold, f1_1, label="Ann. 1", marker="o", linestyle="--")
plt.plot(iou_threshold, f1_2, label="Ann. 2", marker="o", linestyle="--")
plt.xlabel("IoU Threshold", fontweight ='bold', fontsize = 12)
plt.ylabel("F1 score", fontweight ='bold', fontsize = 12)
plt.title("F1 score comparison", fontweight ='bold', fontsize = 15)
plt.legend()
plt.show()



## Plot the annotations of the two raters

In the following we plot some images with the annotations of the two raters to visually compare them. In green we have annotations where raters agree, in red the ones that are unique to rater 1 and in yellow the ones that are unique to rater 2.

In [ ]:
# set plot parameters
from copy import deepcopy

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",  # Computer Modern
    "font.serif": ["cmr10"],   # internal CM font name
    "font.size": 9,
    "figure.dpi": 600
})
width = 3.4  # width in inches

for i in range(len(dataset_1)):
    print(f"Plotting image {i} / {len(dataset_1)}")
    image = dataset_1[i][0].permute(1, 2, 0).numpy()
    bboxes_annotated_1 = dataset_1[i][1]['boxes'].numpy()
    bboxes_annotated_2 = dataset_2[i][1]['boxes'].numpy()
    if len(bboxes_annotated_1) == 0 or len(bboxes_annotated_2) == 0:
        continue
    iou_matrix = compute_iou_matrix(bboxes_annotated_1, bboxes_annotated_2)
    row_ind, col_ind = instance_matcher(deepcopy(iou_matrix), 0.5)

    height = width * image.shape[0] / image.shape[1] # maintain aspect ratio
    plt.figure(figsize=(width, height))
    plt.imshow(image, aspect='equal')
    ax = plt.gca()

    # plot matched boxes in green
    for r, c in zip(row_ind, col_ind):
        box_1 = bboxes_annotated_1[r]
        box_2 = bboxes_annotated_2[c]
        x_min = min(box_1[0], box_2[0])
        y_min = min(box_1[1], box_2[1])
        x_max = max(box_1[2], box_2[2])
        y_max = max(box_1[3], box_2[3])
        rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                             linewidth=0.5, edgecolor='g', facecolor='none', alpha=0.7)
        ax.add_patch(rect)

    # plot unmatched boxes in red (annotator 1) and yellow (annotator 2)
    for r, box in enumerate(bboxes_annotated_1):
        if r not in row_ind:
            x_min, y_min, x_max, y_max = box
            rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                 linewidth=0.5, edgecolor='y', facecolor='none', alpha=0.7)
            ax.add_patch(rect)

    for r, box in enumerate(bboxes_annotated_2):
        if r not in col_ind:
            x_min, y_min, x_max, y_max = box
            rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                 linewidth=0.5, edgecolor='r', facecolor='none', alpha=0.7)
            ax.add_patch(rect)

    # create legend and locate it outside the plot
    green_patch = plt.Line2D([0], [0], color='g', lw=1, label='Matched')
    red_patch = plt.Line2D([0], [0], color='y', lw=1, label='Rater 1 only')
    yellow_patch = plt.Line2D([0], [0], color='r', lw=1, label='Rater 2 only')
    plt.legend(handles=[green_patch, red_patch, yellow_patch], loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)


    plt.axis('off')
    plt.savefig(os.path.join("annotators_comparison", f"image_{i}.pdf"), bbox_inches='tight', pad_inches=0.0, format='pdf')
    plt.show()

# Investigate the size distribution of the prefictions

In [ ]:


iou_threshold = np.linspace(0.1, 0.9, 10)

sizes_1 = []
sizes_2 = []

for annotation_path_1, annotation_path_2 in zip(annotations_1_paths, annotations_2_paths):
    if not os.path.exists(annotation_path_1):
        warnings.warn(f"Annotations path {annotation_path_1} does not exist. Skipping the image.")
        continue
    if not os.path.exists(annotation_path_2):
        warnings.warn(f"Annotations path {annotation_path_2} does not exist. Skipping the image.")
        continue

    # load the annotations
    bboxes_annotated_1 = pd.read_csv(annotation_path_1, index_col=0, sep=",")
    bboxes_annotated_1 = bbox_xyxy_to_box(bboxes_annotated_1)

    bboxes_annotated_2 = pd.read_csv(annotation_path_2, index_col=0, sep=",")
    bboxes_annotated_2 = bbox_xyxy_to_box(bboxes_annotated_2)


    sizes_1_image = []
    sizes_2_image = []

    for bbox in bboxes_annotated_1:
        sizes_1_image.append(bbox.diameter())

    for bbox in bboxes_annotated_2:
        sizes_2_image.append(bbox.diameter())
    
    sizes_1.append(sizes_1_image)
    sizes_2.append(sizes_2_image)
    

In [ ]:
print(precisions_2.shape)
print(len(annotations_1_paths))

Plot the distribution of the sizes

IDEA clould plot the ones having the most significant difference in distribution... -> remember to correct the test 

In [ ]:
sizes_1_flat = [item for sublist in sizes_1 for item in sublist]
sizes_2_flat = [item for sublist in sizes_2 for item in sublist]

fig = plt.subplots(figsize =(12, 8))
plt.hist(sizes_1_flat, bins=30, alpha=0.5, label="Annotator 1")
plt.hist(sizes_2_flat, bins=30, alpha=0.5, label="Annotator 2")
plt.xlabel("Diameter of the bounding box", fontweight ='bold', fontsize = 12)
plt.ylabel("Density", fontweight ='bold', fontsize = 12)
plt.title("Comparison of the bounding box sizes", fontweight ='bold', fontsize = 15)
plt.legend()
plt.show()

fig = plt.subplots(figsize =(12, 8))
plt.hist(sizes_1_flat, bins=50, alpha=0.5, label="Annotator 1", density=True)
plt.hist(sizes_2_flat, bins=50, alpha=0.5, label="Annotator 2", density=True)
plt.xlabel("Diameter of the bounding box", fontweight ='bold', fontsize = 12)
plt.ylabel("Density", fontweight ='bold', fontsize = 12)
plt.title("Comparison of the bounding box sizes (density)", fontweight ='bold', fontsize = 15)
plt.legend()
plt.show()

In [ ]:
from copy import deepcopy
import cv2
from matplotlib.lines import Line2D

iou_threshold = 0.5
predictions = pickle.load(open("predictions.pkl", "rb"))

id = 0
for annotation_path_1, annotation_path_2, (image_norm, meta), bbxes_predicitons in zip(annotations_1_paths, annotations_2_paths, inference_ds, predictions):
    if not os.path.exists(annotation_path_1):
        warnings.warn(f"Annotations path {annotation_path_1} does not exist. Skipping the image.")
        continue
    if not os.path.exists(annotation_path_2):
        warnings.warn(f"Annotations path {annotation_path_2} does not exist. Skipping the image.")
        continue

    # load the annotations
    bboxes_annotated_1 = pd.read_csv(annotation_path_1, index_col=0, sep=",")
    bboxes_annotated_1 = bbox_xyxy_to_box(bboxes_annotated_1)
    nr_annotated_1 = len(bboxes_annotated_1)

    bboxes_annotated_2 = pd.read_csv(annotation_path_2, index_col=0, sep=",")
    bboxes_annotated_2 = bbox_xyxy_to_box(bboxes_annotated_2)
    nr_annotated_2 = len(bboxes_annotated_2)

    # compute the iou matrix
    iou_matrix = compute_iou_matrix(bboxes_annotated_1, bboxes_annotated_2)
    # print("Intersection over union matrix")
    # for i in range(iou_matrix.shape[0]):
    #     for j in range(iou_matrix.shape[1]):
    #         if iou_matrix[i, j] > 0:
    #             print(f"ID: {i} - {j}: {iou_matrix[i, j]}")
    true_positives_1, true_positives_2 = instance_matcher(deepcopy(iou_matrix), iou_threshold)

    # print("True positives")
    # for i, j in zip(true_positives_1, true_positives_2):
    #     print(f"ID: {i} - {j}: {compute_iou(bboxes_annotated_1[i], bboxes_annotated_2[j])}")        
    # compute the false positives
    false_positives_1 = [i for i in range(len(bboxes_annotated_1)) if i not in true_positives_1]
    false_positives_2 = [i for i in range(len(bboxes_annotated_2)) if i not in true_positives_2]

    # load the image
    image_path = meta["path"]
    print(image_path)
    image_1 = cv2.imread(image_path)
    image_2 = cv2.imread(image_path)
    image_3 = cv2.imread(image_path)
    # draw the true positives from the baseline in blue and gree the ones from the annotator
    def from_baseline(bbox, bboes_prediction):
        for b in bboes_prediction:
            if compute_iou(bbox, b) >= 0.5:
                return True
        return False
    
    for bbox in bbxes_predicitons:
        cv2.rectangle(image_3, (bbox.right, bbox.top), (bbox.left, bbox.bottom), (0, 0, 255), 2)

    for i in true_positives_1:
        bbox = bboxes_annotated_1[i]
        color = (0, 0, 255) if from_baseline(bbox, bbxes_predicitons) else (0, 255, 0)
        cv2.rectangle(image_1, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_1, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    for i in true_positives_2:
        bbox = bboxes_annotated_2[i]
        color =  (0, 0, 255) if from_baseline(bbox, bbxes_predicitons) else (0, 255, 0)
        cv2.rectangle(image_2, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_2, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # draw the false positives in red
    for i in false_positives_1:
        bbox = bboxes_annotated_1[i]
        color = (255, 255, 0) if from_baseline(bbox, bbxes_predicitons) else (255, 0, 0)
        cv2.rectangle(image_1, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_1, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    for i in false_positives_2:
        bbox = bboxes_annotated_2[i]
        color = (255, 255, 0) if from_baseline(bbox, bbxes_predicitons) else (255, 0, 0)
        cv2.rectangle(image_2, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_2, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    fig, ax = plt.subplots(1, 3, figsize= (24, 7))
    ax[0].imshow(image_1)
    ax[0].set_title("Annotator 1", fontweight ='bold', fontsize = 12)
    ax[1].imshow(image_2)
    ax[1].set_title("Annotator 2", fontweight ='bold', fontsize = 12)
    ax[2].imshow(image_3)
    ax[2].set_title("Baseline", fontweight ='bold', fontsize = 12)
    fig.suptitle(f"Image {id}", fontweight ='bold', fontsize = 15)
    line_green  = Line2D([0], [0], label='matched annotated', color='green')
    line_blue = Line2D([0], [0], label='matched baseline', color='blue')
    line_red = Line2D([0], [0], label='unmatched annotated', color='red')
    line_yellow = Line2D([0], [0], label='unmatched baseline', color='yellow')
    fig.legend(handles=[line_green, line_blue, line_red, line_yellow])
    plt.show()


    
    id += 1
    


In [ ]:
from copy import deepcopy
import cv2
from matplotlib.lines import Line2D

iou_threshold = 0.5
predictions = pickle.load(open("predictions.pkl", "rb"))

id = 0
for annotation_path_1, annotation_path_2, (image_norm, meta), bbxes_predicitons in zip(annotations_1_paths, annotations_2_paths, inference_ds, predictions):
    if not os.path.exists(annotation_path_1):
        warnings.warn(f"Annotations path {annotation_path_1} does not exist. Skipping the image.")
        continue
    if not os.path.exists(annotation_path_2):
        warnings.warn(f"Annotations path {annotation_path_2} does not exist. Skipping the image.")
        continue

    # load the annotations
    bboxes_annotated_1 = pd.read_csv(annotation_path_1, index_col=0, sep=",")
    bboxes_annotated_1 = bbox_xyxy_to_box(bboxes_annotated_1)
    nr_annotated_1 = len(bboxes_annotated_1)

    bboxes_annotated_2 = pd.read_csv(annotation_path_2, index_col=0, sep=",")
    bboxes_annotated_2 = bbox_xyxy_to_box(bboxes_annotated_2)
    nr_annotated_2 = len(bboxes_annotated_2)

    # compute the iou matrix
    iou_matrix = compute_iou_matrix(bboxes_annotated_1, bboxes_annotated_2)
    # print("Intersection over union matrix")
    # for i in range(iou_matrix.shape[0]):
    #     for j in range(iou_matrix.shape[1]):
    #         if iou_matrix[i, j] > 0:
    #             print(f"ID: {i} - {j}: {iou_matrix[i, j]}")
    true_positives_1, true_positives_2 = instance_matcher(deepcopy(iou_matrix), iou_threshold)

    # print("True positives")
    # for i, j in zip(true_positives_1, true_positives_2):
    #     print(f"ID: {i} - {j}: {compute_iou(bboxes_annotated_1[i], bboxes_annotated_2[j])}")        
    # compute the false positives
    false_positives_1 = [i for i in range(len(bboxes_annotated_1)) if i not in true_positives_1]
    false_positives_2 = [i for i in range(len(bboxes_annotated_2)) if i not in true_positives_2]

    # load the image
    image_path = meta["path"]
    print(image_path)
    image_1 = cv2.imread(image_path)
    image_2 = cv2.imread(image_path)
    

    for i in true_positives_1:
        bbox = bboxes_annotated_1[i]
        color = (0, 255, 0)
        cv2.rectangle(image_1, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_1, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    for i in true_positives_2:
        bbox = bboxes_annotated_2[i]
        color =  (0, 255, 0)
        cv2.rectangle(image_2, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_2, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # draw the false positives in red
    for i in false_positives_1:
        bbox = bboxes_annotated_1[i]
        color = (255, 0, 0)
        cv2.rectangle(image_1, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_1, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    for i in false_positives_2:
        bbox = bboxes_annotated_2[i]
        color = (255, 0, 0)
        cv2.rectangle(image_2, (bbox.right, bbox.top), (bbox.left, bbox.bottom), color, 2)
        cv2.putText(image_2, f"ID: {i}", (bbox.right, bbox.bottom-10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    fig, ax = plt.subplots(1, 2, figsize= (18, 7))
    ax[0].imshow(image_1)
    ax[0].set_title("Annotator 1", fontweight ='bold', fontsize = 12)
    ax[1].imshow(image_2)
    ax[1].set_title("Annotator 2", fontweight ='bold', fontsize = 12)
    fig.suptitle(f"Image {id}", fontweight ='bold', fontsize = 15)
    line_green  = Line2D([0], [0], label='matched annotations', color='green')
    line_red = Line2D([0], [0], label='unmatched annotations', color='red')
    fig.legend(handles=[line_green, line_red])
    plt.show()


    
    id += 1

In [ ]:
import sys
import os
import shutil
# sys.path.append(os.path.dirname(os.getcwd()))
from src.utils.const import *
from src.utils.utils import *

# path = "../dataset/reannotate_dataset.txt"
# old_dataset = "../dataset/merge_05"
# new_dataset_1 = "../dataset/merge_together"
# new_dataset_2 = "../dataset/merge_alone"
# with open(path, "r") as h:
#     files = h.readlines()
#     files = [file.strip() for file in files]

# images_paths = get_images_paths(old_dataset)
# images_paths = [os.path.relpath(image_path, os.path.join(old_dataset, IMAGES_SUBFOLDER)) for image_path in images_paths]
# i = 1
# for image_path in images_paths:
#     annotation_path = image_to_annotations_path(image_path, BBOXES_SUFF)
#     if image_path in files:
#         print(i)
#         i+=1
#         new_dataset = new_dataset_1
#     else:
#         new_dataset = new_dataset_2
#     os.makedirs(os.path.join(new_dataset, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(new_dataset, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(old_dataset, IMAGES_SUBFOLDER, image_path), os.path.join(new_dataset, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(old_dataset, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(new_dataset, ANNOTATIONS_SUBFOLDER, annotation_path))
    



In [ ]:
# import sys
# import os
# import shutil
# import random
# sys.path.append(os.path.dirname(os.getcwd()))
# from src.utils.const import *
# from src.utils.utils import *


# old_dataset = "../dataset/merge_alone"
# new_dataset = "../dataset/merge_alone_1"


# images_paths = get_images_paths(old_dataset)
# print(len(images_paths))
# images_paths = [os.path.relpath(image_path, os.path.join(old_dataset, IMAGES_SUBFOLDER)) for image_path in images_paths]
# random.seed(42)
# sampled_images_paths = random.sample(images_paths, 20) # sample randomly 20 images

# for image_path in sampled_images_paths:
#     print(image_path)
#     annotation_path = image_to_annotations_path(image_path, BBOXES_SUFF)
#     os.makedirs(os.path.join(new_dataset, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(new_dataset, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(old_dataset, IMAGES_SUBFOLDER, image_path), os.path.join(new_dataset, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(old_dataset, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(new_dataset, ANNOTATIONS_SUBFOLDER, annotation_path))

In [ ]:
# import sys
# import os
# import shutil
# import random
# sys.path.append(os.path.dirname(os.getcwd()))
# from src.utils.const import *
# from src.utils.utils import *


# dataset = "../dataset/merge" # all the images
# dataset_1 = "../dataset/merge_together" # the images that are corrected together
# dataset_2 = "../dataset/merge_alone_1" # the images that are corrected alone


# images_path = get_images_paths(dataset)
# images_path = [os.path.relpath(image_path, os.path.join(dataset, IMAGES_SUBFOLDER)) for image_path in images_path]
# images_path_1 = get_images_paths(dataset_1)
# images_path_1 = [os.path.relpath(image_path, os.path.join(dataset_1, IMAGES_SUBFOLDER)) for image_path in images_path_1]
# images_path_2 = get_images_paths(dataset_2)
# images_path_2 = [os.path.relpath(image_path, os.path.join(dataset_2, IMAGES_SUBFOLDER)) for image_path in images_path_2]

# images_path_remaining = list(set(images_path) - set(images_path_1) - set(images_path_2))

# random.seed(42)
# sampled_images_1 = random.sample(images_path_remaining, 30) # sample randomly 30 images
# sampled_images_2 = list(set(images_path_remaining) - set(sampled_images_1))
# print(len(sampled_images_1))
# print(len(sampled_images_2))

# dataset_new_1 = "../dataset/merge_alone_2_LT"
# for image_path in sampled_images_1:
#     annotation_path = image_to_annotations_path(image_path, BBOXES_SUFF)
#     os.makedirs(os.path.join(dataset_new_1, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(dataset_new_1, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(dataset, IMAGES_SUBFOLDER, image_path), os.path.join(dataset_new_1, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(dataset, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(dataset_new_1, ANNOTATIONS_SUBFOLDER, annotation_path))

# dataset_new_2 = "../dataset/merge_alone_2_EK"
# for image_path in sampled_images_2:
#     annotation_path = image_to_annotations_path(image_path, BBOXES_SUFF)
#     os.makedirs(os.path.join(dataset_new_2, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(dataset_new_2, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(dataset, IMAGES_SUBFOLDER, image_path), os.path.join(dataset_new_2, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(dataset, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(dataset_new_2, ANNOTATIONS_SUBFOLDER, annotation_path))

In [ ]:
# import sys
# import os
# import shutil
# import random
# sys.path.append(os.path.dirname(os.getcwd()))
# from src.utils.const import *
# from src.utils.utils import *


# dataset = "../dataset/merge_alone_1_corrected_LT" 
# dataset_1 = "../dataset/Lucrezia" 
# dataset_1_dest = "../dataset/merge_alone_1_LT" 
# dataset_2 = "../dataset/Ekin"
# dataset_2_dest = "../dataset/merge_alone_1_EK" 


# images_path = get_images_paths(dataset)
# images_path = [os.path.relpath(image_path, os.path.join(dataset, IMAGES_SUBFOLDER)) for image_path in images_path]

# for image_path in images_path:
#     annotation_path = image_to_annotations_path(image_path, BBOXES_SUFF)
#     os.makedirs(os.path.join(dataset_1_dest, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(dataset_1_dest, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(dataset_1, IMAGES_SUBFOLDER, image_path), os.path.join(dataset_1_dest, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(dataset_1, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(dataset_1_dest, ANNOTATIONS_SUBFOLDER, annotation_path))

#     os.makedirs(os.path.join(dataset_2_dest, IMAGES_SUBFOLDER, os.path.dirname(image_path)), exist_ok=True)
#     os.makedirs(os.path.join(dataset_2_dest, ANNOTATIONS_SUBFOLDER, os.path.dirname(annotation_path)), exist_ok=True)
#     shutil.copyfile(os.path.join(dataset_2, IMAGES_SUBFOLDER, image_path), os.path.join(dataset_2_dest, IMAGES_SUBFOLDER, image_path))
#     shutil.copyfile(os.path.join(dataset_2, ANNOTATIONS_SUBFOLDER, annotation_path), os.path.join(dataset_2_dest, ANNOTATIONS_SUBFOLDER, annotation_path))